<a href="https://colab.research.google.com/github/Samu24042/CienciaDeDatos/blob/main/icd_taller5_limpieza_datos.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# **¡¡ ANTES DE EMPEZAR !!**

Deshabilita autocompletar con IA. Esta acción te ayudará a aprender de verdad. Si estás en Colab sigue estos pasos:

1.  Ir a Herramientas \ Configuración \ Asistencia de IA
2.  Desactivar la casilla **"Mostrar autocompletado impulsado por IA"**
3.  Activar la casilla **"Ocultar funciones de IA generativa"**

# **MÓDULO 2: LIMPIEZA DE DATOS CON PANDAS**

La materia prima en cualquier proyecto de ciencia de datos o *Machine Learning* son, precisamente, los datos. Sin embargo, en la vida real, los datos rara vez vienen listos para ser analizados. Usualmente, una vez recolectados, nos damos cuenta de que tienen registros incompletos, filas repetidas, valores extremos o errores tipográficos.

De hecho, mucho del trabajo de un científico de datos se dedica específicamente a la limpieza de datos. Un modelo matemático o estadístico alimentado con datos basura producirá resultados basura (*Garbage In, Garbage Out*).

**2.1 Carga del Dataset:**
Para este módulo usaremos un *dataset* real y muy famoso: los registros de pasajeros del Titanic. Contiene datos demográficos y si sobrevivieron o no al naufragio. Cargaremos los datos directamente desde un repositorio público.

In [25]:
# --- IMPORTAR PANDAS Y CARGAR DATOS ---

import pandas as pd

# URL pública del dataset crudo
url_titanic = "https://raw.githubusercontent.com/datasciencedojo/datasets/master/titanic.csv"

# Leer el archivo CSV
df_titanic = pd.read_csv(url_titanic)

# Vamos a crear un par de filas duplicadas artificialmente para propósitos de la clase
df_titanic = pd.concat([df_titanic, df_titanic.iloc[[0, 1]]], ignore_index=True)

# Veamos una muestra aleatoria de 5 filas
display(df_titanic.sample(5))


,PassengerId,Survived,Pclass,Name,Sex,Age,SibSp,Parch,Ticket,Fare,Cabin,Embarked
3,4,1,1,"Futrelle, Mrs. Jacques Heath (Lily May Peel)",female,35.0,1,0,113803,53.1000,C123,S
795,796,0,2,"Otter, Mr. Richard",male,39.0,0,0,28213,13.0000,NaN,S
528,529,0,3,"Salonen, Mr. Johan Werner",male,39.0,0,0,3101296,7.9250,NaN,S
125,126,1,3,"Nicola-Yarred, Master. Elias",male,12.0,1,0,2651,11.2417,NaN,C
238,239,0,2,"Pengelly, Mr. Frederick William",male,19.0,0,0,28665,10.5000,NaN,S


**2.2 Registros Duplicados:**
El primer paso suele ser verificar si tenemos filas idénticas repetidas en nuestro *dataset*. Entrenar un modelo pasándole la misma información dos veces puede generar sesgos. Pandas nos permite identificar y eliminar estas redundancias fácilmente.

In [26]:
# --- ELIMINAR DUPLICADOS ---

# 1. Contar cuántas filas duplicadas existen
# .duplicated() devuelve True o False. Al sumarlas, suma los True (1).
num_duplicados = df_titanic.duplicated().sum()
print(f"Se encontraron {num_duplicados} filas duplicadas.")

# 2. Eliminar los duplicados
# drop_duplicates() elimina las repeticiones. inplace=True aplica el cambio directamente al DataFrame original.
df_titanic.drop_duplicates(inplace=True)

print(f"Dimensiones después de limpiar duplicados: {df_titanic.shape}")


Se encontraron 2 filas duplicadas.
Dimensiones después de limpiar duplicados: (891, 12)


**2.3 Columnas Irrelevantes o con Demasiados Nulos:**
Algunas columnas pueden no aportar información útil para nuestro problema de negocio o técnico. Por ejemplo, el identificador único del pasajero o el número de cabina (si la mayoría de la gente no lo tiene registrado). Si una columna tiene más del 70% de datos faltantes, suele ser mejor eliminarla.

In [27]:
# --- INSPECCIÓN DE NULOS Y ELIMINACIÓN DE COLUMNAS ---

# isna().sum() nos muestra cuántos valores faltantes hay por cada columna
print("--- Valores nulos por columna ---")
print(df_titanic.isna().sum())

# La columna 'Cabin' tiene casi 700 datos faltantes. La eliminaremos junto con 'PassengerId' (que no aporta valor predictivo).
# Usamos drop() e indicamos los nombres de las columnas en una lista.
columnas_a_borrar = ['PassengerId', 'Cabin']
df_titanic.drop(columns=columnas_a_borrar, inplace=True)

print("\nColumnas restantes:")
print(df_titanic.columns)


--- Valores nulos por columna ---
PassengerId      0
Survived         0
Pclass           0
Name             0
Sex              0
Age            177
SibSp            0
Parch            0
Ticket           0
Fare             0
Cabin          687
Embarked         2
dtype: int64

Columnas restantes:
Index(['Survived', 'Pclass', 'Name', 'Sex', 'Age', 'SibSp', 'Parch', 'Ticket',
       'Fare', 'Embarked'],
      dtype='object')


**2.4 Tratamiento de Datos Faltantes (NaN):**
La mayoría de los modelos matemáticos no pueden procesar valores nulos (celdas vacías o `NaN`). Tenemos dos opciones principales:
1. **Eliminar la fila completa:** Si los datos faltantes son muy pocos y no perdemos información vital.
2. **Imputar el valor:** Rellenar el espacio con un valor estadístico, como el promedio (media) o la mediana de esa columna.

*Nota:* Se prefiere la **mediana** cuando la columna tiene valores extremos (outliers), ya que el promedio se deja afectar fácilmente por ellos.

In [28]:
# --- MANEJO DE NaNs (NOT A NUMBER) ---

# CASO 1: Eliminar filas.
# La columna 'Embarked' (puerto de embarque) solo tiene 2 nulos. Eliminaremos esas dos filas enteras.
# dropna(subset=...) borra filas si hay nulos en las columnas especificadas.
df_titanic.dropna(subset=['Embarked'], inplace=True)

# CASO 2: Imputar valores (Rellenar).
# La columna 'Age' tiene muchos nulos. Reemplazaremos los nulos con la mediana de las edades.
mediana_edad = df_titanic['Age'].median()
df_titanic['Age'] = df_titanic['Age'].fillna(mediana_edad) # fillna() rellena los espacios vacíos

print("Verificación de nulos después de la limpieza:")
print(df_titanic.isna().sum())


Verificación de nulos después de la limpieza:
Survived    0
Pclass      0
Name        0
Sex         0
Age         0
SibSp       0
Parch       0
Ticket      0
Fare        0
Embarked    0
dtype: int64


**2.5 Errores Tipográficos y Estandarización de Texto:**
En variables categóricas ingresadas manualmente, es común encontrar errores como usar mayúsculas/minúsculas de forma inconsistente, o espacios extra. Para que Pandas no interprete "Ingeniería" e "ingeniería" como dos categorías distintas, debemos estandarizar.

In [29]:
# --- ESTANDARIZACIÓN DE TEXTO ---

# Para aplicar operaciones de texto (strings) a toda una columna, usamos la propiedad .str

# 1. Convertir todo a minúsculas en la columna 'Name'
df_titanic['Name'] = df_titanic['Name'].str.lower()

# 2. Convertir la variable categórica 'Sex' ('male', 'female') a formato título ('Male', 'Female' - primera letra mayúscula)
df_titanic['Sex'] = df_titanic['Sex'].str.title()

display(df_titanic[['Name', 'Sex']].head(3))


,Name,Sex
0,"braund, mr. owen harris",Male
1,"cumings, mrs. john bradley (florence briggs th...",Female
2,"heikkinen, miss. laina",Female


**2.6 Transformaciones Avanzadas - Funciones Lambda:**
En Python, usualmente definimos funciones usando la palabra reservada `def` cuando vamos a repetir un proceso complejo muchas veces. Sin embargo, en la limpieza de datos, a menudo necesitamos realizar una operación matemática o lógica sencilla que solo usaremos una vez. Para esto resulta mucho más eficiente usar **Funciones Lambda**.

Una función lambda es una función "anónima" (no necesita un nombre formal) que se escribe en una sola línea de código. En Pandas, las Funciones Lambda son muy útiles al combinarlas con el método `.apply()`. Esto nos permite recorrer cada celda de una columna aplicándole la función. Es muy parecido a si tuviéramos un ciclo `for`, pero optimizado para datos.

* **Estructura tradicional:** `def mi_funcion(x): return x * 2`
* **Estructura Lambda:** `lambda x: x * 2`

**Sintaxis básica con condicionales:** `lambda variable: <Resultado_Si_Verdadero> if <Condición> else <Resultado_Si_Falso>`

In [30]:
# --- USO DE FUNCIONES LAMBDA ---

# OBJETIVO: Crear una columna que clasifique a los pasajeros en 'Adulto' o 'Menor'.
# Usamos .apply() para que la lambda tome la 'Age' de cada fila individualmente y la evalúe.

df_titanic['Etapa_Vida'] = df_titanic['Age'].apply(lambda edad: 'Adulto' if edad >= 18 else 'Menor')
display(df_titanic[['Name', 'Age', 'Etapa_Vida']].head(4))


,Name,Age,Etapa_Vida
0,"braund, mr. owen harris",22.0,Adulto
1,"cumings, mrs. john bradley (florence briggs th...",38.0,Adulto
2,"heikkinen, miss. laina",26.0,Adulto
3,"futrelle, mrs. jacques heath (lily may peel)",35.0,Adulto


**2.7 Extracción de Patrones - Expresiones Regulares (Regex):**
Imaginemos que necesitamos extraer números de teléfono de una columna de texto. No se puede buscar un texto exacto porque cada número es distinto, pero sí se puede buscar el *patrón* (ej. tres dígitos, un guion, siete dígitos). Las **Expresiones Regulares (Regex)** son un lenguaje universal utilizado en casi todos los lenguajes de programación para definir y buscar estos patrones en cadenas de texto.

En Pandas, combinamos Regex con el método `.str.extract(patron)` para sacar información que está "escondida" dentro del texto.

Algunos comodines y cuantificadores básicos de Regex:
* `\d` : Encuentra cualquier dígito numérico (del 0 al 9).
* `[A-Za-z]` : Encuentra cualquier letra, sin importar si es mayúscula o minúscula.
* `+` : Es un cuantificador. Indica que el elemento anterior puede repetirse una o más veces continuas (ej. `\d+` buscará el número '105' completo, no solo el '1').
* `\.` : En Regex, el punto normalmente significa "cualquier carácter". Si queremos buscar un punto real (ortográfico), debemos usar la barra invertida.

In [31]:
# --- USO DE EXPRESIONES REGULARES (REGEX) ---

# OBJETIVO: Extraer el título de la persona (Mr., Mrs., Miss., etc.) que está dentro del nombre.
# El patrón r' ([A-Za-z]+)\.' le dice a la máquina exactamente lo siguiente:
#   1. ' '          -> Busca un espacio en blanco inicial.
#   2. ([A-Za-z]+)  -> Busca un grupo de letras continuas (esto es lo que extraerá).
#   3. \.           -> Asegúrate de que esas letras terminen en un punto literal.

# expand=False es un parámetro técnico para que Pandas nos devuelva una Columna simple (Serie) y no una tabla.
df_titanic['Titulo'] = df_titanic['Name'].str.extract(r' ([A-Za-z]+)\.', expand=False)

# Veamos cómo extrajo los títulos
display(df_titanic[['Name', 'Titulo']].head(5))

# Un truco útil: ver cuántos pasajeros hay por cada título extraído
print("\nConteo de títulos encontrados:")
print(df_titanic['Titulo'].value_counts())


,Name,Titulo
0,"braund, mr. owen harris",mr
1,"cumings, mrs. john bradley (florence briggs th...",mrs
2,"heikkinen, miss. laina",miss
3,"futrelle, mrs. jacques heath (lily may peel)",mrs
4,"allen, mr. william henry",mr



Conteo de títulos encontrados:
Titulo
mr          517
miss        181
mrs         124
master       40
dr            7
rev           6
col           2
mlle          2
major         2
ms            1
mme           1
don           1
lady          1
sir           1
capt          1
countess      1
jonkheer      1
Name: count, dtype: int64


# **RETOS: MÓDULO 2**

Vamos a analizar las lecturas de una red de sensores IoT ubicados en una planta industrial. Los sensores miden temperatura y nivel de vibración de varias máquinas. Sin embargo, el archivo crudo extraído de la base de datos contiene todo tipo de errores de captura.

**Primero, ejecuta la siguiente celda para generar el DataFrame de trabajo. NO modifiques esta celda.**

In [32]:
# --- DATOS DE ENTRADA (NO MODIFICAR) ---
import pandas as pd
import numpy as np

# Generación de dataset crudo simulado
datos_crudos = {
    'id_sensor': ['S-101', 'S-102', 's-103', 'S-101', 'S-104', 'S-105', 'S-105', 'S-106', 'S-107', 'S-108'],
    'tipo_maquina': ['Bomba', 'Compresor', 'BOMBA', 'Bomba', 'Ventilador', 'Compresor', 'Compresor', 'Turbina', 'Bomba', 'Ventilador'],
    'temperatura_c': [45.2, np.nan, 46.1, 45.2, 55.4, 1050.0, 1050.0, np.nan, 43.5, 58.9], # El 1050.0 es un outlier obvio
    'vibracion_rms': [1.2, 2.5, 1.4, 1.2, 3.1, 4.5, 4.5, np.nan, 1.1, -2.5], # Valores negativos en vibración son ruido
    'estado_conexion': ['Online', 'Offline', 'Online', 'Online', 'Offline ', 'Online', 'Online', 'Offline', 'Online', 'Online']
}

df_sensores = pd.DataFrame(datos_crudos)

print("--- DataFrame Crudo ---")
display(df_sensores)


--- DataFrame Crudo ---


,id_sensor,tipo_maquina,temperatura_c,vibracion_rms,estado_conexion
0,S-101,Bomba,45.2,1.2,Online
1,S-102,Compresor,NaN,2.5,Offline
2,s-103,BOMBA,46.1,1.4,Online
3,S-101,Bomba,45.2,1.2,Online
4,S-104,Ventilador,55.4,3.1,Offline
5,S-105,Compresor,1050.0,4.5,Online
6,S-105,Compresor,1050.0,4.5,Online
7,S-106,Turbina,NaN,NaN,Offline
8,S-107,Bomba,43.5,1.1,Online
9,S-108,Ventilador,58.9,-2.5,Online


## Problema 1. Registros Duplicados y Tipografía
Hay sensores que reportaron la misma lectura dos veces. Además, la columna `tipo_maquina` tiene inconsistencias en minúsculas/mayúsculas ("Bomba" vs "BOMBA").

**Entregable:** 1. Elimina las filas exactas que estén duplicadas.
2. Estandariza la columna `tipo_maquina` para que todo el texto esté en mayúsculas.
3. Estandariza la columna `id_sensor` para que todo esté en mayúsculas.

In [33]:
# --- CÓDIGO PROBLEMA 1 ---

df_sensores = df_sensores.drop_duplicates()
df_sensores["tipo_maquina"] = df_sensores["tipo_maquina"].str.upper()
df_sensores["id_sensor"] = df_sensores["id_sensor"].str.upper()

display(df_sensores)

,id_sensor,tipo_maquina,temperatura_c,vibracion_rms,estado_conexion
0,S-101,BOMBA,45.2,1.2,Online
1,S-102,COMPRESOR,NaN,2.5,Offline
2,S-103,BOMBA,46.1,1.4,Online
4,S-104,VENTILADOR,55.4,3.1,Offline
5,S-105,COMPRESOR,1050.0,4.5,Online
7,S-106,TURBINA,NaN,NaN,Offline
8,S-107,BOMBA,43.5,1.1,Online
9,S-108,VENTILADOR,58.9,-2.5,Online


## Problema 2. Manejo de Variables Faltantes (NaN)
La columna de `temperatura_c` tiene algunos datos faltantes. Al tratarse de máquinas continuas, se te pide que no borres estas filas.

**Entregable:** Rellena los valores nulos (`NaN`) de la columna `temperatura_c` utilizando el **promedio** (media) de las temperaturas registradas. Para la columna `vibracion_rms`, elimina la fila si el dato es nulo.

In [34]:
# --- CÓDIGO PROBLEMA 2 ---

media_temp = df_sensores["temperatura_c"].mean()
df_sensores["temperatura_c"] = df_sensores["temperatura_c"].fillna(media_temp)
df_sensores = df_sensores.dropna(subset=["vibracion_rms"])
display(df_sensores)

,id_sensor,tipo_maquina,temperatura_c,vibracion_rms,estado_conexion
0,S-101,BOMBA,45.200000,1.2,Online
1,S-102,COMPRESOR,216.516667,2.5,Offline
2,S-103,BOMBA,46.100000,1.4,Online
4,S-104,VENTILADOR,55.400000,3.1,Offline
5,S-105,COMPRESOR,1050.000000,4.5,Online
8,S-107,BOMBA,43.500000,1.1,Online
9,S-108,VENTILADOR,58.900000,-2.5,Online


## Problema 3. Identificación y Filtro de Outliers (Valores Extremos)
Un sensor falló y marcó una temperatura de más de 1000 °C (un valor irreal para el proceso). Por otro lado, la vibración RMS ($RMS = \sqrt{\frac{x^2}{n}}$) representa magnitudes físicas, por lo que **no puede ser un número negativo**.

**Entregable:** Filtra el DataFrame para que **solo** queden las filas que cumplan **ambas** condiciones lógicas:
1. La temperatura debe ser menor a 150 °C.
2. La vibración RMS debe ser un valor positivo (mayor o igual a 0).

In [35]:
# --- CÓDIGO PROBLEMA 3 ---

df_sensores = df_sensores[
    (df_sensores["temperatura_c"] < 150) &
    (df_sensores["vibracion_rms"] >= 0)
]

display(df_sensores)

,id_sensor,tipo_maquina,temperatura_c,vibracion_rms,estado_conexion
0,S-101,BOMBA,45.2,1.2,Online
2,S-103,BOMBA,46.1,1.4,Online
4,S-104,VENTILADOR,55.4,3.1,Offline
8,S-107,BOMBA,43.5,1.1,Online


## Problema 4. Limpieza de Espacios Invisibles y Alertas
Al revisar la columna `estado_conexion`, parece estar bien, pero los equipos 'Offline' tienen espacios en blanco invisibles al final del texto (ej. `\"Offline \"`). Además, los operadores necesitan una alerta visual rápida sobre el nivel de vibración.

**Entregable:** 1. Utiliza el método `.str.strip()` para quitar los espacios en blanco sobrantes en la columna `estado_conexion`.
2. Utiliza una **función lambda** y el método `.apply()` en la columna `vibracion_rms` para crear una nueva columna llamada `alerta_vibracion`. Si la vibración es mayor a `3.0`, debe decir `'Alta'`, de lo contrario, `'Normal'`.
3. Muestra el DataFrame completamente limpio y final.

In [36]:
# --- CÓDIGO PROBLEMA 4 ---

df_sensores["estado_conexion"] = (
    df_sensores["estado_conexion"].str.strip()
)
df_sensores["alerta_vibracion"] = (
    df_sensores["vibracion_rms"]
    .apply(lambda x: "Alta" if x > 3.0 else "Normal")
)
display(df_sensores)

,id_sensor,tipo_maquina,temperatura_c,vibracion_rms,estado_conexion,alerta_vibracion
0,S-101,BOMBA,45.2,1.2,Online,Normal
2,S-103,BOMBA,46.1,1.4,Online,Normal
4,S-104,VENTILADOR,55.4,3.1,Offline,Alta
8,S-107,BOMBA,43.5,1.1,Online,Normal
